## Data Collectors API Endpoints Test

**IMPORTANT: Backend 서버를 사전에 기동해야 합니다.**

```bash
# Terminal에서 실행:
uvicorn src.app.api.main:app --reload
```

Day 3 작업 중 FastAPI Collectors 엔드포인트 기능 테스트

테스트 대상 엔드포인트:
- `GET /api/collectors/sources` - 사용 가능한 소스 목록 조회
- `POST /api/collectors/search` - 여러 소스에서 동시 검색
- `POST /api/collectors/arxiv` - arXiv 논문 검색
- `POST /api/collectors/news` - 뉴스 기사 검색

In [1]:
import json
import requests
from typing import Dict, List

# API base URL
BASE_URL = "http://127.0.0.1:8000"
COLLECTORS_BASE = f"{BASE_URL}/api/collectors"

print("✓ Setup complete")

✓ Setup complete


In [2]:
# Server health check
try:
    response = requests.get(f"{BASE_URL}/health", timeout=5.0)
    if response.status_code == 200:
        print("✅ Server is running and healthy!")
        print(f"Response: {response.json()}")
    else:
        print(f"⚠️ Server responded with status code: {response.status_code}")
except requests.exceptions.ConnectionError:
    print("❌ Cannot connect to server. Please start the backend:")
    print("   uvicorn src.app.api.main:app --reload")
except Exception as e:
    print(f"❌ Health check failed: {e}")

✅ Server is running and healthy!
Response: {'status': 'healthy'}


### 1. List Available Sources

`GET /api/collectors/sources` - 지원되는 데이터 소스 목록 조회

In [3]:
# 소스 목록 조회
response = requests.get(f"{COLLECTORS_BASE}/sources")

print(f"Status Code: {response.status_code}")
print(f"\nAvailable Data Sources:")
print("=" * 60)

result = response.json()
sources = result['sources']

for source in sources:
    print(f"\n📚 {source['name'].upper()}")
    print(f"   Type: {source['type']}")
    print(f"   Description: {source['description']}")
    print(f"   Supported Filters: {', '.join(source['supported_filters'])}")

Status Code: 200

Available Data Sources:

📚 ARXIV
   Type: paper
   Description: Academic papers from arXiv.org
   Supported Filters: categories, sort_by, sort_order

📚 NEWS
   Type: news
   Description: Tech and AI news articles
   Supported Filters: domains, date_filter, freshness


### 2. Search arXiv Papers

`POST /api/collectors/arxiv` - arXiv에서 학술 논문 검색

In [4]:
# 기본 arXiv 검색
request_data = {
    "query": "large language model",
    "limit": 5
}

response = requests.post(f"{COLLECTORS_BASE}/arxiv", json=request_data)

print(f"Status Code: {response.status_code}")
result = response.json()

print(f"\nArXiv Search Results:")
print(f"Total: {result['total']} papers")
print(f"Errors: {result['errors']}")
print("\n" + "=" * 60)

for i, paper in enumerate(result['results'], 1):
    print(f"\n{i}. {paper['title']}")
    print(f"   Source: {paper['source_name']} ({paper['source_type']})")
    print(f"   URL: {paper['url']}")
    print(f"   Authors: {', '.join(paper['metadata']['authors'][:3])}...")
    print(f"   Category: {paper['metadata']['primary_category']}")
    print(f"   Published: {paper['metadata']['published'][:10]}")
    print(f"   Abstract: {paper['content'][:100]}...")

Status Code: 200

ArXiv Search Results:
Total: 5 papers
Errors: []


1. Learning From Failure: Integrating Negative Examples when Fine-tuning Large Language Models as Agents
   Source: arXiv (paper)
   URL: http://arxiv.org/abs/2402.11651v2
   Authors: Renxi Wang, Haonan Li, Xudong Han...
   Category: cs.CL
   Published: 2024-02-18
   Abstract: Large language models (LLMs) have achieved success in acting as agents, which interact with environm...

2. Demystifying Instruction Mixing for Fine-tuning Large Language Models
   Source: arXiv (paper)
   URL: http://arxiv.org/abs/2312.10793v3
   Authors: Renxi Wang, Haonan Li, Minghao Wu...
   Category: cs.CL
   Published: 2023-12-17
   Abstract: Instruction tuning significantly enhances the performance of large language models (LLMs) across var...

3. WizardLM: Empowering large pre-trained language models to follow complex instructions
   Source: arXiv (paper)
   URL: http://arxiv.org/abs/2304.12244v3
   Authors: Can Xu, Qingfeng Sun, Kai Zhe

### 3. Search arXiv with Filters

카테고리 필터링 및 정렬 옵션 사용

In [5]:
# 필터링된 arXiv 검색
request_data = {
    "query": "attention mechanism",
    "limit": 3,
    "filters": {
        "categories": ["cs.AI", "cs.LG"],
        "sort_by": "relevance",
        "sort_order": "descending"
    }
}

response = requests.post(f"{COLLECTORS_BASE}/arxiv", json=request_data)
result = response.json()

print(f"Filtered ArXiv Search (cs.AI, cs.LG only):")
print(f"Total: {result['total']} papers\n")

for i, paper in enumerate(result['results'], 1):
    print(f"{i}. {paper['title']}")
    print(f"   Categories: {', '.join(paper['metadata']['categories'])}")
    print(f"   arXiv ID: {paper['metadata']['arxiv_id']}")
    print(f"   PDF: {paper['metadata']['pdf_url']}")
    print()

Filtered ArXiv Search (cs.AI, cs.LG only):
Total: 3 papers

1. Déjà vu: A Contextualized Temporal Attention Mechanism for Sequential Recommendation
   Categories: cs.IR, cs.CL, cs.LG
   arXiv ID: 2002.00741v1
   PDF: https://arxiv.org/pdf/2002.00741v1

2. Pay Attention to What You Need
   Categories: cs.CL, cs.AI
   arXiv ID: 2307.13365v3
   PDF: https://arxiv.org/pdf/2307.13365v3

3. Benign Overfitting in Token Selection of Attention Mechanism
   Categories: cs.LG
   arXiv ID: 2409.17625v3
   PDF: https://arxiv.org/pdf/2409.17625v3



### 4. Search News Articles

`POST /api/collectors/news` - 뉴스 기사 검색

**Note**: Serper API 키가 필요합니다. 키가 없거나 만료된 경우 에러가 발생할 수 있습니다.

In [6]:
# 기본 뉴스 검색
try:
    request_data = {
        "query": "artificial intelligence",
        "limit": 5
    }

    response = requests.post(f"{COLLECTORS_BASE}/news", json=request_data)
    
    if response.status_code == 200:
        result = response.json()
        
        print(f"News Search Results:")
        print(f"Total: {result['total']} articles")
        print(f"Errors: {result['errors']}")
        print("\n" + "=" * 60)
        
        for i, article in enumerate(result['results'], 1):
            print(f"\n{i}. {article['title']}")
            print(f"   Source: {article['source_name']} ({article['source_type']})")
            print(f"   URL: {article['url']}")
            print(f"   Published: {article['metadata'].get('published_date', 'N/A')}")
            print(f"   Snippet: {article['content'][:100]}...")
    else:
        print(f"❌ Request failed with status code: {response.status_code}")
        print(f"Error: {response.json()}")
        
except Exception as e:
    print(f"❌ News search failed: {e}")
    print("\nNote: This may be due to:")
    print("  1. Missing or invalid SERPER_API_KEY in .env")
    print("  2. Serper API rate limit exceeded")
    print("  3. Network connectivity issues")

News Search Results:
Total: 10 articles
Errors: []


1. Is language the same as intelligence? The AI industry desperately needs it to be
   Source: News (news)
   URL: https://www.theverge.com/ai-artificial-intelligence/827820/large-language-models-ai-intelligence-neuroscience-problems
   Published: 3 weeks ago
   Snippet: Neuroscience indicates language is distinct from thought, raising questions about whether AI large l...

2. OpenAI, Anthropic, and Block Are Teaming Up to Make AI Agents Play Nice
   Source: News (news)
   URL: https://www.wired.com/story/openai-anthropic-and-block-are-teaming-up-on-ai-agent-standards/
   Published: 4 days ago
   Snippet: OpenAI, Anthropic, and Block have cofounded a new open source organization—the Agentic AI Foundation...

3. Trump Signs Executive Order That Threatens to Punish States for Passing AI Laws
   Source: News (news)
   URL: https://www.wired.com/story/trump-signs-executive-order-ai-state-laws/
   Published: 2 days ago
   Snippet: The ord

### 5. Search News with Domain Filters

특정 도메인만 필터링하여 뉴스 검색

In [7]:
# 도메인 필터링된 뉴스 검색
try:
    request_data = {
        "query": "ChatGPT",
        "limit": 3,
        "filters": {
            "domains": ["techcrunch.com", "venturebeat.com"],
            "date_filter": "w"  # 최근 1주일
        }
    }

    response = requests.post(f"{COLLECTORS_BASE}/news", json=request_data)
    
    if response.status_code == 200:
        result = response.json()
        
        print(f"Filtered News (TechCrunch & VentureBeat):")
        print(f"Total: {result['total']} articles\n")
        
        for i, article in enumerate(result['results'], 1):
            print(f"{i}. {article['title']}")
            print(f"   Source: {article['metadata'].get('source_name', 'N/A')}")
            print(f"   URL: {article['url']}")
            print()
    else:
        print(f"Request failed: {response.status_code}")
        print(response.json())
        
except Exception as e:
    print(f"Filtered news search failed: {e}")

Filtered News (TechCrunch & VentureBeat):
Total: 10 articles

1. OpenAI fires back at Google with GPT-5.2 after ‘code red’ memo
   Source: TechCrunch
   URL: https://techcrunch.com/2025/12/11/openai-fires-back-at-google-with-gpt-5-2-after-code-red-memo/

2. GPT-5.2 first impressions: a powerful update, especially for business tasks and workflows
   Source: VentureBeat
   URL: https://venturebeat.com/ai/gpt-5-2-first-impressions-a-powerful-update-especially-for-business-tasks

3. OpenAI's GPT-5.2 is here: what enterprises need to know
   Source: VentureBeat
   URL: https://venturebeat.com/ai/openais-gpt-5-2-is-here-what-enterprises-need-to-know

4. Google launches sub-$5 AI Plus plan in India to compete with ChatGPT Go
   Source: TechCrunch
   URL: https://techcrunch.com/2025/12/10/google-launches-sub-5-ai-plus-plan-in-india-to-compete-with-chatgpt-go/

5. OpenAI report reveals a 6x productivity gap between AI power users and everyone else
   Source: VentureBeat
   URL: https://ventureb

### 6. Multi-Source Search

`POST /api/collectors/search` - 여러 소스에서 동시 검색

In [8]:
# 여러 소스에서 동시 검색
request_data = {
    "query": "transformer",
    "sources": ["arxiv"],  # arxiv만 검색 (news는 API 키 필요)
    "limit": 3
}

response = requests.post(f"{COLLECTORS_BASE}/search", json=request_data)
result = response.json()

print(f"Multi-Source Search Results:")
print(f"Total: {result['total']} items")
print(f"Errors: {result['errors']}")
print("\n" + "=" * 60)

# 소스 타입별로 그룹화
papers = [r for r in result['results'] if r['source_type'] == 'paper']
news = [r for r in result['results'] if r['source_type'] == 'news']

print(f"\n📄 Papers: {len(papers)}")
for i, paper in enumerate(papers, 1):
    print(f"{i}. {paper['title'][:60]}...")
    print(f"   [{paper['source_name']}] {paper['url']}")

print(f"\n📰 News: {len(news)}")
for i, article in enumerate(news, 1):
    print(f"{i}. {article['title'][:60]}...")
    print(f"   [{article['source_name']}] {article['url']}")

Multi-Source Search Results:
Total: 3 items
Errors: []


📄 Papers: 3
1. PyramidTNT: Improved Transformer-in-Transformer Baselines wi...
   [arXiv] http://arxiv.org/abs/2201.00978v1
2. Learning to Cluster Faces via Transformer...
   [arXiv] http://arxiv.org/abs/2104.11502v1
3. MLP Can Be A Good Transformer Learner...
   [arXiv] http://arxiv.org/abs/2404.05657v1

📰 News: 0


### 7. Search All Sources

소스를 지정하지 않으면 모든 소스에서 검색

In [9]:
# 모든 소스에서 검색 (sources 파라미터 생략)
request_data = {
    "query": "GPT-4",
    "limit": 2  # 각 소스당 2개씩
}

response = requests.post(f"{COLLECTORS_BASE}/search", json=request_data)
result = response.json()

print(f"Search All Sources Results:")
print(f"Total: {result['total']} items")
print(f"Errors: {len(result['errors'])} errors")

if result['errors']:
    print("\n⚠️ Errors occurred:")
    for error in result['errors']:
        print(f"  - {error}")

print("\n" + "=" * 60)
print("All Results:")
for i, item in enumerate(result['results'], 1):
    print(f"\n{i}. [{item['source_type'].upper()}] {item['title'][:50]}...")
    print(f"   Source: {item['source_name']}")
    print(f"   URL: {item['url']}")

Search All Sources Results:
Total: 12 items
Errors: 0 errors

All Results:

1. [PAPER] Instruction Tuning with GPT-4...
   Source: arXiv
   URL: http://arxiv.org/abs/2304.03277v1

2. [PAPER] Capabilities of GPT-4 on Medical Challenge Problem...
   Source: arXiv
   URL: http://arxiv.org/abs/2303.13375v2

3. [NEWS] OpenAI fires back at Google with GPT-5.2 after ‘co...
   Source: News
   URL: https://techcrunch.com/2025/12/11/openai-fires-back-at-google-with-gpt-5-2-after-code-red-memo/

4. [NEWS] 4 things Claude AI can do that ChatGPT can't...
   Source: News
   URL: https://www.zdnet.com/article/4-things-claude-ai-can-do-that-chatgpt-cant/

5. [NEWS] Large language models can do jaw-dropping things. ...
   Source: News
   URL: https://www.technologyreview.com/2024/03/04/1089403/large-language-models-amazing-but-nobody-knows-why/

6. [NEWS] Mistral launches powerful Devstral 2 coding model ...
   Source: News
   URL: https://venturebeat.com/ai/mistral-launches-powerful-devstral-2-coding-

### 8. Error Handling Test

잘못된 요청에 대한 에러 처리 확인

In [10]:
# 1. 빈 쿼리 테스트
print("Test 1: Empty query")
try:
    request_data = {
        "query": "",
        "limit": 5
    }
    response = requests.post(f"{COLLECTORS_BASE}/arxiv", json=request_data)
    print(f"Status: {response.status_code}")
    if response.status_code != 200:
        print(f"✓ Validation error: {response.json()}")
    else:
        print(f"Result: {response.json()['total']} items (API may handle empty query)")
except Exception as e:
    print(f"✓ Error caught: {type(e).__name__}")

# 2. 잘못된 소스 이름
print("\nTest 2: Invalid source name")
try:
    request_data = {
        "query": "test",
        "sources": ["invalid_source"],
        "limit": 5
    }
    response = requests.post(f"{COLLECTORS_BASE}/search", json=request_data)
    result = response.json()
    print(f"Status: {response.status_code}")
    print(f"Total: {result['total']}")
    print(f"Errors: {result['errors']}")
    if "Unknown source" in str(result['errors']):
        print("✓ Invalid source error detected")
except Exception as e:
    print(f"Error: {e}")

# 3. 필수 필드 누락
print("\nTest 3: Missing required field (query)")
try:
    request_data = {
        "limit": 5
        # query 필드 누락
    }
    response = requests.post(f"{COLLECTORS_BASE}/arxiv", json=request_data)
    print(f"Status: {response.status_code}")
    if response.status_code == 422:
        print(f"✓ Validation error (422): {response.json()['detail'][0]['msg']}")
except Exception as e:
    print(f"Error: {e}")

# 4. 잘못된 limit 값
print("\nTest 4: Invalid limit value (negative)")
try:
    request_data = {
        "query": "test",
        "limit": -1
    }
    response = requests.post(f"{COLLECTORS_BASE}/arxiv", json=request_data)
    print(f"Status: {response.status_code}")
    if response.status_code == 422:
        print(f"✓ Validation error detected")
        print(f"Detail: {response.json()['detail'][0]}")
    else:
        print(f"Result: {response.json()}")
except Exception as e:
    print(f"Error: {e}")

Test 1: Empty query
Status: 422
✓ Validation error: {'detail': [{'type': 'string_too_short', 'loc': ['body', 'query'], 'msg': 'String should have at least 1 character', 'input': '', 'ctx': {'min_length': 1}}]}

Test 2: Invalid source name
Status: 500
Error: 'total'

Test 3: Missing required field (query)
Status: 422
✓ Validation error (422): Field required

Test 4: Invalid limit value (negative)
Status: 422
✓ Validation error detected
Detail: {'type': 'greater_than_equal', 'loc': ['body', 'limit'], 'msg': 'Input should be greater than or equal to 1', 'input': -1, 'ctx': {'ge': 1}}


### 9. Response Schema Validation

API 응답이 올바른 스키마를 따르는지 검증

In [11]:
def validate_collection_response(response_data: dict) -> dict:
    """CollectionResponse 스키마 검증"""
    errors = []
    
    # 필수 필드 검증
    required_fields = ['total', 'results', 'errors']
    for field in required_fields:
        if field not in response_data:
            errors.append(f"Missing required field: {field}")
    
    # total 타입 검증
    if not isinstance(response_data.get('total'), int):
        errors.append("'total' must be an integer")
    
    # results 타입 검증
    if not isinstance(response_data.get('results'), list):
        errors.append("'results' must be a list")
    
    # errors 타입 검증
    if not isinstance(response_data.get('errors'), list):
        errors.append("'errors' must be a list")
    
    # 각 result 아이템 검증
    if isinstance(response_data.get('results'), list):
        required_item_fields = ['title', 'content', 'url', 'source_type', 'source_name', 'metadata', 'collected_at']
        for i, item in enumerate(response_data['results']):
            for field in required_item_fields:
                if field not in item:
                    errors.append(f"Result item {i}: Missing field '{field}'")
    
    return {
        "valid": len(errors) == 0,
        "errors": errors
    }

# 스키마 검증 실행
request_data = {
    "query": "test",
    "limit": 2
}

response = requests.post(f"{COLLECTORS_BASE}/arxiv", json=request_data)
result = response.json()

validation_result = validate_collection_response(result)

print("Response Schema Validation:")
print("=" * 60)
if validation_result['valid']:
    print("✅ All schema validations passed!")
    print(f"\nResponse structure:")
    print(f"  - total: {result['total']} (int)")
    print(f"  - results: {len(result['results'])} items (list)")
    print(f"  - errors: {len(result['errors'])} errors (list)")
    if result['results']:
        print(f"\nFirst result item fields:")
        for key in result['results'][0].keys():
            print(f"  - {key}")
else:
    print("❌ Schema validation errors:")
    for error in validation_result['errors']:
        print(f"  - {error}")

Response Schema Validation:
✅ All schema validations passed!

Response structure:
  - total: 2 (int)
  - results: 2 items (list)
  - errors: 0 errors (list)

First result item fields:
  - title
  - content
  - url
  - source_type
  - source_name
  - metadata
  - collected_at


### 10. Performance Test

API 응답 시간 측정

In [12]:
import time

def benchmark_endpoint(url: str, data: dict, name: str) -> dict:
    """API 엔드포인트 성능 측정"""
    start = time.time()
    response = requests.post(url, json=data)
    elapsed = time.time() - start
    
    result = response.json() if response.status_code == 200 else {}
    
    return {
        "name": name,
        "status_code": response.status_code,
        "response_time": elapsed,
        "total_items": result.get('total', 0),
        "has_errors": len(result.get('errors', [])) > 0
    }

# 벤치마크 실행
test_query = "deep learning"
test_limit = 5

benchmarks = []

# arXiv 엔드포인트
benchmarks.append(benchmark_endpoint(
    f"{COLLECTORS_BASE}/arxiv",
    {"query": test_query, "limit": test_limit},
    "POST /arxiv"
))

# Multi-source 엔드포인트 (arxiv만)
benchmarks.append(benchmark_endpoint(
    f"{COLLECTORS_BASE}/search",
    {"query": test_query, "sources": ["arxiv"], "limit": test_limit},
    "POST /search (arxiv only)"
))

print("Performance Benchmark Results:")
print("=" * 60)
for bench in benchmarks:
    print(f"\n{bench['name']}:")
    print(f"  Status: {bench['status_code']}")
    print(f"  Response time: {bench['response_time']:.2f}s")
    print(f"  Items collected: {bench['total_items']}")
    print(f"  Has errors: {bench['has_errors']}")
    if bench['total_items'] > 0:
        print(f"  Avg time per item: {bench['response_time'] / bench['total_items']:.2f}s")

Performance Benchmark Results:

POST /arxiv:
  Status: 200
  Response time: 0.18s
  Items collected: 5
  Has errors: False
  Avg time per item: 0.04s

POST /search (arxiv only):
  Status: 200
  Response time: 0.05s
  Items collected: 5
  Has errors: False
  Avg time per item: 0.01s


### 11. End-to-End Workflow Test

실제 사용 시나리오: 소스 조회 → 검색 → 결과 처리

In [13]:
# 1. 사용 가능한 소스 조회
print("Step 1: Get available sources")
sources_response = requests.get(f"{COLLECTORS_BASE}/sources")
sources = sources_response.json()['sources']
available_sources = [s['name'] for s in sources]
print(f"Available sources: {available_sources}")

# 2. arXiv에서 검색 (API 키 불필요)
print("\nStep 2: Search arXiv for papers")
search_query = "neural network"
arxiv_response = requests.post(
    f"{COLLECTORS_BASE}/arxiv",
    json={"query": search_query, "limit": 3}
)
arxiv_results = arxiv_response.json()
print(f"Found {arxiv_results['total']} papers")

# 3. 결과 처리 및 분석
print("\nStep 3: Process and analyze results")
if arxiv_results['total'] > 0:
    papers = arxiv_results['results']
    
    # 카테고리 분포 분석
    categories = {}
    for paper in papers:
        cat = paper['metadata']['primary_category']
        categories[cat] = categories.get(cat, 0) + 1
    
    print(f"\nCategory distribution:")
    for cat, count in categories.items():
        print(f"  {cat}: {count}")
    
    # 최신 논문 찾기
    most_recent = max(papers, key=lambda p: p['metadata']['published'])
    print(f"\nMost recent paper:")
    print(f"  Title: {most_recent['title']}")
    print(f"  Published: {most_recent['metadata']['published'][:10]}")
    print(f"  URL: {most_recent['url']}")
    
    # 평균 요약 길이
    avg_content_length = sum(len(p['content']) for p in papers) / len(papers)
    print(f"\nAverage abstract length: {avg_content_length:.0f} characters")

print("\n" + "=" * 60)
print("✅ End-to-end workflow completed successfully!")

Step 1: Get available sources
Available sources: ['arxiv', 'news']

Step 2: Search arXiv for papers
Found 3 papers

Step 3: Process and analyze results

Category distribution:
  cs.NE: 1
  cs.LG: 1
  stat.ML: 1

Most recent paper:
  Title: Predicting concentration levels of air pollutants by transfer learning and recurrent neural network
  Published: 2025-01-30
  URL: http://arxiv.org/abs/2502.01654v1

Average abstract length: 1148 characters

✅ End-to-end workflow completed successfully!


### 12. Partial Failure Test

일부 소스가 실패해도 다른 소스의 결과는 반환되는지 확인

In [14]:
# 여러 소스 검색 (일부는 실패할 수 있음)
request_data = {
    "query": "GPT",
    "sources": ["arxiv", "news"],  # news는 API 키 필요
    "limit": 2
}

response = requests.post(f"{COLLECTORS_BASE}/search", json=request_data)
result = response.json()

print("Partial Failure Test:")
print("=" * 60)
print(f"Status Code: {response.status_code}")
print(f"Total items: {result['total']}")
print(f"Errors: {len(result['errors'])}")

if result['errors']:
    print("\n⚠️ Errors encountered:")
    for error in result['errors']:
        print(f"  - {error}")

if result['total'] > 0:
    print(f"\n✅ Partial success: {result['total']} items collected despite errors")
    
    # 소스별 결과 개수
    source_counts = {}
    for item in result['results']:
        source = item['source_name']
        source_counts[source] = source_counts.get(source, 0) + 1
    
    print("\nResults by source:")
    for source, count in source_counts.items():
        print(f"  {source}: {count}")
else:
    print("\n❌ All sources failed")

Partial Failure Test:
Status Code: 200
Total items: 12
Errors: 0

✅ Partial success: 12 items collected despite errors

Results by source:
  arXiv: 2
  News: 10


### Summary

이 노트북에서 다룬 내용:

#### Basic API Tests (1-3)
1. ✓ List available sources (GET /sources)
2. ✓ Basic arXiv search (POST /arxiv)
3. ✓ Filtered arXiv search with categories

#### News API Tests (4-5)
4. ✓ Basic news search (POST /news)
5. ✓ Domain-filtered news search

#### Multi-Source Tests (6-7)
6. ✓ Multi-source search (POST /search)
7. ✓ Search all sources (default behavior)

#### Quality & Validation (8-9)
8. ✓ Error handling (empty query, invalid params)
9. ✓ Response schema validation

#### Performance & Integration (10-12)
10. ✓ Performance benchmarks
11. ✓ End-to-end workflow
12. ✓ Partial failure handling